# 对LLM基本代码使用用法的小结

## 基本环境的准备

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, PreTrainedModel, modeling_outputs
import torch
from loguru import logger
import os
import sys
from typing import cast

os.environ["HTTP_PROXY"] = "http://127.0.0.1:6382"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:6382"

logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True)

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

## 基础总结1：使用 GPT-2 完成下一个 Token 预测与句子生成

In [ ]:
class GPT2GenerateToken:
    def __init__(self, tokenizer: GPT2Tokenizer, model: PreTrainedModel) -> None:
        self._tokenizer: GPT2Tokenizer = tokenizer
        self._model: PreTrainedModel = model
        self._top_k: int = 3  # 默认topk设置为3

    def generate_next_token(self, prompt: str) -> None:
        """
        模型生成基本的下一个token
        :param prompt: 提示词字符串
        :return: None
        """
        logger.info(f"Step1: 从提示词字符串->对应的token id列表->对应的张量类形式->分词后的tokens列表")
        logger.info(f"prompt : {prompt}")
        input_ids: list[int] = self._tokenizer.encode(prompt)
        logger.info(f"对应的token ids ：{input_ids}")
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)
        logger.info(f"对应的张量类的表示：{token_ids}")
        tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids]
        logger.info(f"对应的分开的词的形式：{tokens}")

        with torch.no_grad():
            output: modeling_outputs.CausalLMOutputWithCrossAttentions = self._model(token_ids)
            logits: torch.Tensor = cast(torch.Tensor, output.logits)
            logger.info(
                f"logits张量的shape为：{logits.shape}，其中的第一个维度是batch，第二个维度是token数，第三个维度是词表数")
            next_token_logits: torch.Tensor = logits[0, -1, :]
            logger.info(f"用于打分的logits维度是{next_token_logits.shape}\n值为{next_token_logits}")
            next_token_probabilities: torch.Tensor = torch.softmax(next_token_logits, dim=0)
            logger.info(f"经过softmax后的概率shape：{next_token_probabilities.shape}，概率是{next_token_probabilities}")
            top_k_probabilities, top_k_index = torch.topk(next_token_probabilities, self._top_k)
            top_k_tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in top_k_index]
            for i, (probability, index, token) in enumerate(zip(top_k_probabilities, top_k_index, top_k_tokens)):
                logger.info(f"Top {i} : probability = {probability}, index = {index}, token = {token}")

    def generate_sentence(self, prompt: str, max_token_num: int = 50) -> str:
        """
        模型进行基本的续写
        :param max_token_num: 最大句子长度限制
        :param prompt: 提示词字符串
        :return: 续写的字符串
        """
        # process basic input
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        # logits and probabilities, use non-greedy settings
        should_generate_end: bool = False
        while not should_generate_end:
            with torch.no_grad():
                logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
                probabilities: torch.Tensor = torch.softmax(logits, dim=0)
                next_token_id: int = cast(int, torch.argmax(probabilities).item())
                next_token: str = cast(str, self._tokenizer.decode(next_token_id))

                prompt += next_token
                token_ids = torch.cat([token_ids, torch.tensor([[next_token_id]], dtype=torch.long)], dim=1)

                if next_token in [".", "?", "!"] or token_ids.shape[1] >= max_token_num:
                    should_generate_end = True
        return prompt

展示GPT-2生成下一个token的详细过程

In [ ]:
gpt2_generate_token = GPT2GenerateToken(gpt2_tokenizer, gpt2_model)
gpt2_generate_token.generate_next_token("Thank you very")

使用GPT-2进行句子续写，使用greedy settings

In [ ]:
answer: str = gpt2_generate_token.generate_sentence("The meaning of life is", max_token_num=50)
logger.info(answer)

## 基础总结2：GPT-2 对下一个 Token 的打分与采样过程

增加温度以及non-greedy的考虑

In [ ]:
class GPT2GenerateTokenWithTemperature(GPT2GenerateToken):
    """
    考虑温度的打分过程
    """

    def __init__(self, tokenizer: GPT2Tokenizer, model: PreTrainedModel, temperature: float) -> None:
        super().__init__(tokenizer, model)
        self._temperature = temperature

    @property
    def temperature(self) -> float:
        return self._temperature

    @temperature.setter
    def temperature(self, value: float) -> None:
        if value <= 0:
            raise ValueError("temperature must be greater than 0")
        self._temperature = value

    def generate_next_token(self, prompt: str, is_greedy: bool = False) -> str:
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        with torch.no_grad():
            logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
            scaled_logits: torch.Tensor = logits / self._temperature
            token_probabilities: torch.Tensor = torch.softmax(scaled_logits, dim=0)

            if is_greedy:
                next_token, _ = self._get_greedy_next_token_with_id(token_probabilities)
            else:
                next_token, _ = self._get_non_greedy_next_token_with_id(token_probabilities)
            return next_token

    def generate_sentence(self, prompt: str, is_greedy: bool = False, max_token_num: int = 50) -> str:
        """
        模型进行基本的续写
        :param is_greedy: 是否采用greedy settings
        :param max_token_num: 最大句子长度限制
        :param prompt: 提示词字符串
        :return: 续写的字符串
        """
        # process basic input
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        # logits and probabilities, use non-greedy settings
        should_generate_end: bool = False
        while not should_generate_end:
            with torch.no_grad():
                logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
                logits: torch.Tensor = logits / self._temperature
                token_probabilities: torch.Tensor = torch.softmax(logits, dim=0)
                if is_greedy:
                    next_token, next_token_id = self._get_greedy_next_token_with_id(token_probabilities)
                else:
                    next_token, next_token_id = self._get_non_greedy_next_token_with_id(token_probabilities)

                prompt += next_token
                token_ids = torch.cat([token_ids, torch.tensor([[next_token_id]], dtype=torch.long)], dim=1)

                if next_token in [".", "?", "!"] or token_ids.shape[1] >= max_token_num:
                    should_generate_end = True
        return prompt

    def _get_greedy_next_token_with_id(self, probabilities: torch.Tensor) -> tuple[str, int]:
        next_token_id: int = cast(int, torch.argmax(probabilities).item())
        return cast(str, self._tokenizer.decode(next_token_id)), next_token_id

    def _get_non_greedy_next_token_with_id(self, probabilities: torch.Tensor) -> tuple[str, int]:
        next_token_id: int = cast(int, torch.multinomial(probabilities, 1).item())
        return cast(str, self._tokenizer.decode(next_token_id)), next_token_id


对不同温度以及是否是non-greedy的情况进行展示

In [ ]:
gpt2_generate_token_with_temperature = GPT2GenerateTokenWithTemperature(gpt2_tokenizer, gpt2_model, 1.0)
logger.info("test case 1 : temperature = 1.0, non-greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is")
logger.info(input_prompt)
logger.info('=' * 20)

logger.info("test case 2 : temperature = 1.0, greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is", is_greedy=True)
logger.info(input_prompt)
logger.info('=' * 20)

gpt2_generate_token_with_temperature.temperature = 0.3
logger.info("test case 3 : temperature = 0.3, non-greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is")
logger.info(input_prompt)
logger.info('=' * 20)

logger.info("test case 4 : temperature = 0.3, greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is", is_greedy=True)
logger.info(input_prompt)
logger.info('=' * 20)

gpt2_generate_token_with_temperature.temperature = 1.5
logger.info("test case 5 : temperature = 1.5, non-greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is")
logger.info(input_prompt)
logger.info('=' * 20)

logger.info("test case 6 : temperature = 1.5, greedy settings")
input_prompt = gpt2_generate_token_with_temperature.generate_sentence("The meaning of life is", is_greedy=True)
logger.info(input_prompt)
logger.info('=' * 20)

## 基础总结3：从文本到 Token、Embedding 与上下文 Feature

In [ ]:
class GPT2TokenEmbeddingFeature:
    """
    用于展示GPT-2模型token对应的embedding以及feature的流程中的行为
    """

    def __init__(self, tokenizer: GPT2Tokenizer, model: GPT2LMHeadModel) -> None:
        self._tokenizer = tokenizer
        self._model = model
        self._wte_embedding: torch.Tensor = model.transformer.wte.weight.detach()
        self._wpe_embedding: torch.Tensor = model.transformer.wpe.weight.detach()

    def display_embedding_and_feature(self, prompt: str) -> None:
        """
        展示prompt在前向传播过程中形成embedding，feature的过程
        :param prompt:
        :return:
        """
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)
        tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids]

        with torch.no_grad():
            outputs = self._model(token_ids, output_hidden_states=True)
        hidden_states = outputs.hidden_states

        logger.info(f"input token nums = {len(tokens)}")
        for position, (token, token_id) in enumerate(zip(tokens, input_ids)):
            logger.info(f"{'=' * 20} token_{position} = {token!r} {'*' * 20}")
            wte: torch.Tensor = self._wte_embedding[token_id]
            wpe: torch.Tensor = self._wpe_embedding[position]
            logger.info(f"WTE[:3] = {wte[:3].tolist()}")
            logger.info(f"WPE[:3] = {wpe[:3].tolist()}")

            embedding: torch.Tensor = wte + wpe
            logger.info(f"embedding = {embedding[:3].tolist()}")

            hidden_state_zero: torch.Tensor = hidden_states[0][0, position, :]
            logger.info(f"hidden_state_zero = {hidden_state_zero[:3].tolist()}")
            logger.info(f" hidden_state_zero = embedding_vector !")
            hidden_state_low: torch.Tensor = hidden_states[1][0, position, :]
            hidden_state_mid: torch.Tensor = hidden_states[6][0, position, :]
            hidden_state_high: torch.Tensor = hidden_states[-1][0, position, :]
            logger.info(f"hidden_state_low[:3] = {hidden_state_low[:3].tolist()}")
            logger.info(f"hidden_state_mid[:3] = {hidden_state_mid[:3].tolist()}")
            logger.info(f"hidden_state_high[:3] = {hidden_state_high[:3].tolist()}")


下面的代码展示了从token到embedding再到各层feature的过程。其中embedding（wte+wpe）和feature的第0层是一致的。

In [ ]:
gpt2_token_embedding_feature = GPT2TokenEmbeddingFeature(gpt2_tokenizer, cast(GPT2LMHeadModel, gpt2_model))
gpt2_token_embedding_feature.display_embedding_and_feature("I went to the bank")

## 基础总结4：WTE 与 WPE 如何共同构成 Transformer 的输入

本部分以同一个词在不同的位置、以及同一个位置的不同含义的词为例来展示transformer过程中对相同含义不同位置、相同位置不同含义的词的处理方式

In [ ]:
class GPT2AnalogyAndWPE(GPT2TokenEmbeddingFeature):
    """
    用于展示GPT-2模型同一个token在不同位置以及同义词的区分情况
    """

    def __init__(self, tokenizer: GPT2Tokenizer, model: GPT2LMHeadModel) -> None:
        super().__init__(tokenizer, model)

    def display_wte_wpe_feature(self, prompt1: str, prompt2: str, target: str) -> None:
        """
        同一个 token、同一个语义，在不同position下，先看WPE如何改变输入embedding，再看这种差异经过Transformer后如何体现在feature上。
        :param prompt1: 提示词1
        :param prompt2: 提示词2
        :param target: 用于展示的目标词
        :return: None
        """
        input_ids1: list[int] = self._tokenizer.encode(prompt1)
        input_ids2: list[int] = self._tokenizer.encode(prompt2)
        token_ids1: torch.Tensor = torch.tensor([input_ids1], dtype=torch.long)
        token_ids2: torch.Tensor = torch.tensor([input_ids2], dtype=torch.long)
        tokens1: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids1]
        tokens2: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids2]

        pos1: int = next(i for i in range(len(tokens1)) if tokens1[i].strip().lower() == target.strip().lower())
        pos2: int = next(i for i in range(len(tokens2)) if tokens2[i].strip().lower() == target.strip().lower())
        wte1: torch.Tensor = self._wte_embedding[input_ids1[pos1]]
        wte2: torch.Tensor = self._wte_embedding[input_ids2[pos2]]
        wpe1: torch.Tensor = self._wpe_embedding[pos1]
        wpe2: torch.Tensor = self._wpe_embedding[pos2]
        embedding1: torch.Tensor = wte1 + wpe1
        embedding2: torch.Tensor = wte2 + wpe2

        logger.info(f"wte1[:3] = {wte1[:3].tolist()}")
        logger.info(f"wte2[:3] = {wte2[:3].tolist()}")
        logger.info(f"wpe1[:3] = {wpe1[:3].tolist()}")
        logger.info(f"wpe2[:3] = {wpe2[:3].tolist()}")
        logger.info(f"embedding1[:3] = {embedding1[:3].tolist()}")
        logger.info(f"embedding2[:3] = {embedding2[:3].tolist()}")

        with torch.no_grad():
            output1 = self._model(token_ids1, output_hidden_states=True).hidden_states
            output2 = self._model(token_ids2, output_hidden_states=True).hidden_states

        feature_low1 = output1[1][0, pos1, :]
        feature_low2 = output2[1][0, pos2, :]
        logger.info(f"feature_low1[:3] = {feature_low1[:3].tolist()}")
        logger.info(f"feature_low2[:3] = {feature_low2[:3].tolist()}")

        feature_mid1 = output1[5][0, pos1, :]
        feature_mid2 = output2[5][0, pos2, :]
        logger.info(f"feature_mid1[:3] = {feature_mid1[:3].tolist()}")
        logger.info(f"feature_mid2[:3] = {feature_mid2[:3].tolist()}")

        feature_high1 = output1[-1][0, pos1, :]
        feature_high2 = output2[-1][0, pos2, :]
        logger.info(f"feature_high1[:3] = {feature_high1[:3].tolist()}")
        logger.info(f"feature_high2[:3] = {feature_high2[:3].tolist()}")


首先展示同一个token在不同的位置时引起的差异

In [ ]:
gpt2_analogy_and_wpe = GPT2AnalogyAndWPE(gpt2_tokenizer, cast(GPT2LMHeadModel, gpt2_model))
gpt2_analogy_and_wpe.display_wte_wpe_feature("I love my cat because they are cute", "Their cat is not happy today",
                                             "cat")

再展示同一个词（甚至可以位于同一个位置），但含义不同时所展示出的差异

In [ ]:
gpt2_analogy_and_wpe.display_wte_wpe_feature("I deposited money at the bank yesterday",
                                             "I saw water near the bank last week", "bank")